In [1]:
# ============================================================
# FRAGDENSTAAT:
# FORTSETZBARER DOKUMENT-DOWNLOADER MIT KORREKTEN DATENTYPEN
# ============================================================
#
# Dieser Code:
# - erkennt den Projekt- und Datenordner automatisch
# - lädt eine vorhandene Statusdatei oder die Download-Queue
# - korrigiert die Datentypen aller Statusspalten
# - lädt zunächst maximal 100 Dokumente
# - überspringt bereits vorhandene Dateien
# - speichert regelmäßig den Fortschritt
# - protokolliert Fehler
# - kann nach einem Abbruch fortgesetzt werden
#
# Es wird zunächst mit 100 Dokumenten getestet.
# Für alle Dokumente später:
# MAX_DOWNLOADS_PRO_LAUF = None
# ============================================================

import re
import time
from pathlib import Path
from urllib.parse import unquote, urlparse

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# 1. EINSTELLUNGEN
# ============================================================

# Zunächst maximal 100 Dokumente laden.
# Für einen vollständigen Lauf später auf None setzen.
MAX_DOWNLOADS_PRO_LAUF = 100

# Pause zwischen zwei Dateien
PAUSE_SECONDS = 0.5

# Nach jeweils so vielen Dokumenten speichern
SAVE_EVERY = 10

# Maximale Dateigröße pro Dokument in MB.
# None bedeutet keine Größenbegrenzung.
MAX_FILE_SIZE_MB = None

# Verbindungs- und Lesetimeout
TIMEOUT = (20, 180)


# ============================================================
# 2. ORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

if ARBEITSORDNER.name.lower() == "datenbank":
    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENORDNER = ARBEITSORDNER

elif (ARBEITSORDNER / "Datenbank").is_dir():
    PROJEKTORDNER = ARBEITSORDNER
    DATENORDNER = ARBEITSORDNER / "Datenbank"

else:
    raise FileNotFoundError(
        "\nDer Ordner 'Datenbank' konnte nicht gefunden werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}"
    )


DOKUMENTORDNER = PROJEKTORDNER / "Dokumente"
DOKUMENTORDNER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. DATEIPFADE
# ============================================================

QUEUE_CSV = DATENORDNER / "fragdenstaat_download_queue_all.csv"
QUEUE_JSONL = DATENORDNER / "fragdenstaat_download_queue_all.jsonl"

STATUS_CSV = DATENORDNER / "fragdenstaat_download_status.csv"
FEHLER_CSV = DATENORDNER / "fragdenstaat_download_fehler.csv"


print("=" * 78)
print("FRAGDENSTAAT-DOKUMENT-DOWNLOADER")
print("=" * 78)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nDatenordner:")
print(DATENORDNER)

print("\nDokumentordner:")
print(DOKUMENTORDNER)


# ============================================================
# 4. DOWNLOAD-QUEUE ODER STATUSDATEI LADEN
# ============================================================

if STATUS_CSV.exists():

    print("\nVorhandene Statusdatei wird geladen ...")

    queue_df = pd.read_csv(
        STATUS_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    verwendete_datei = STATUS_CSV

elif QUEUE_CSV.exists():

    print("\nDownload-Queue wird geladen ...")

    queue_df = pd.read_csv(
        QUEUE_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    verwendete_datei = QUEUE_CSV

elif QUEUE_JSONL.exists():

    print("\nCSV nicht gefunden. JSONL wird geladen ...")

    queue_df = pd.read_json(
        QUEUE_JSONL,
        lines=True,
    )

    verwendete_datei = QUEUE_JSONL

else:

    raise FileNotFoundError(
        "\nKeine Download-Queue gefunden.\n\n"
        "Erwartete Dateien:\n"
        f"- {QUEUE_CSV}\n"
        f"- {QUEUE_JSONL}"
    )


print("\nGeladene Datei:")
print(verwendete_datei)

print("\nDatensätze:")
print(f"{len(queue_df):,}")


# ============================================================
# 5. PFLICHTSPALTEN PRÜFEN
# ============================================================

PFLICHTSPALTEN = [
    "id",
    "title",
    "file_url",
]

fehlende_spalten = [
    spalte
    for spalte in PFLICHTSPALTEN
    if spalte not in queue_df.columns
]

if fehlende_spalten:
    raise KeyError(
        "\nFolgende Pflichtspalten fehlen:\n"
        + "\n".join(
            f"- {spalte}"
            for spalte in fehlende_spalten
        )
    )


# ============================================================
# 6. STATUSSPALTEN ERSTELLEN
# ============================================================

standardwerte = {
    "download_status": "pending",
    "download_attempts": 0,
    "downloaded_at": "",
    "local_file_path": "",
    "http_status": "",
    "download_error": "",
    "actual_file_size": "",
    "content_type": "",
}

for spalte, standardwert in standardwerte.items():
    if spalte not in queue_df.columns:
        queue_df[spalte] = standardwert


# ============================================================
# 7. DATENTYPEN AUSDRÜCKLICH KORRIGIEREN
# ============================================================

# Zahlenspalten
queue_df["id"] = pd.to_numeric(
    queue_df["id"],
    errors="coerce",
)

queue_df = queue_df[
    queue_df["id"].notna()
].copy()

queue_df["id"] = queue_df["id"].astype("int64")

queue_df["download_attempts"] = pd.to_numeric(
    queue_df["download_attempts"],
    errors="coerce",
).fillna(0).astype("int64")


# Alle Spalten, in die später Text geschrieben wird,
# werden ausdrücklich als object/string behandelt.
TEXTSPALTEN = [
    "title",
    "file_url",
    "download_status",
    "downloaded_at",
    "local_file_path",
    "http_status",
    "download_error",
    "actual_file_size",
    "content_type",
]

for spalte in TEXTSPALTEN:

    queue_df[spalte] = (
        queue_df[spalte]
        .fillna("")
        .astype("object")
    )


# Leere Statuswerte normalisieren
queue_df.loc[
    queue_df["download_status"].astype(str).str.strip().eq(""),
    "download_status",
] = "pending"


print("\nDatentypen der Statusspalten wurden korrigiert.")


# ============================================================
# 8. HTTP-SESSION MIT WIEDERHOLUNGSVERSUCHEN
# ============================================================

retry_strategy = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=2,
    status_forcelist=(
        429,
        500,
        502,
        503,
        504,
    ),
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
    raise_on_status=False,
)

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "OpenLens-Capstone/0.1 "
            "(educational document processing)"
        ),
        "Accept": "*/*",
    }
)

adapter = HTTPAdapter(
    max_retries=retry_strategy,
    pool_connections=2,
    pool_maxsize=2,
)

session.mount("https://", adapter)
session.mount("http://", adapter)


# ============================================================
# 9. HILFSFUNKTIONEN
# ============================================================

def sicherer_dateiname(text: str) -> str:
    """
    Entfernt unter Windows ungültige Zeichen aus Dateinamen.
    """

    text = unquote(str(text))

    text = re.sub(
        r'[<>:"/\\|?*\x00-\x1f]',
        "_",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    text = text.rstrip(". ")

    if not text:
        return "dokument"

    return text[:180]


def dateiname_aus_url(
    url: str,
    dokument_id: int,
    titel: str,
) -> str:
    """
    Erzeugt aus ID und URL einen stabilen lokalen Dateinamen.
    """

    url_pfad = urlparse(str(url)).path

    original_name = Path(
        unquote(url_pfad)
    ).name

    if not original_name:
        original_name = str(titel)

    original_name = sicherer_dateiname(
        original_name
    )

    if not Path(original_name).suffix:
        original_name += ".bin"

    return f"{int(dokument_id)}_{original_name}"


def status_speichern() -> None:
    """
    Speichert die gesamte Queue mit dem aktuellen Status.
    """

    queue_df.to_csv(
        STATUS_CSV,
        index=False,
        encoding="utf-8-sig",
    )


def fehler_speichern() -> None:
    """
    Speichert nur fehlgeschlagene Downloads.
    """

    fehler_df = queue_df[
        queue_df["download_status"].eq("failed")
    ].copy()

    fehler_df.to_csv(
        FEHLER_CSV,
        index=False,
        encoding="utf-8-sig",
    )


def dokument_herunterladen(
    index: int,
    row: pd.Series,
) -> bool:
    """
    Lädt genau ein Dokument herunter und aktualisiert den Status.
    """

    dokument_id = int(row["id"])
    titel = str(row["title"])
    file_url = str(row["file_url"]).strip()

    dateiname = dateiname_aus_url(
        url=file_url,
        dokument_id=dokument_id,
        titel=titel,
    )

    zielpfad = DOKUMENTORDNER / dateiname

    aktueller_versuch = int(
        queue_df.at[
            index,
            "download_attempts",
        ]
    )

    queue_df.at[
        index,
        "download_attempts",
    ] = aktueller_versuch + 1


    # Bereits vorhandene Datei übernehmen
    if zielpfad.exists() and zielpfad.stat().st_size > 0:

        queue_df.at[
            index,
            "download_status",
        ] = "downloaded"

        queue_df.at[
            index,
            "downloaded_at",
        ] = str(
            pd.Timestamp.now(tz="UTC").isoformat()
        )

        queue_df.at[
            index,
            "local_file_path",
        ] = str(zielpfad)

        queue_df.at[
            index,
            "actual_file_size",
        ] = str(zielpfad.stat().st_size)

        queue_df.at[
            index,
            "download_error",
        ] = ""

        return True


    temporaerer_pfad = zielpfad.with_suffix(
        zielpfad.suffix + ".part"
    )

    try:

        with session.get(
            file_url,
            stream=True,
            timeout=TIMEOUT,
            allow_redirects=True,
        ) as response:

            queue_df.at[
                index,
                "http_status",
            ] = str(response.status_code)

            queue_df.at[
                index,
                "content_type",
            ] = str(
                response.headers.get(
                    "Content-Type",
                    "",
                )
            )

            response.raise_for_status()

            content_length = response.headers.get(
                "Content-Length"
            )

            if (
                MAX_FILE_SIZE_MB is not None
                and content_length
            ):

                erwartete_mb = (
                    int(content_length)
                    / (1024 ** 2)
                )

                if erwartete_mb > MAX_FILE_SIZE_MB:

                    queue_df.at[
                        index,
                        "download_status",
                    ] = "skipped_too_large"

                    queue_df.at[
                        index,
                        "download_error",
                    ] = (
                        f"Dateigröße {erwartete_mb:.2f} MB; "
                        f"Maximum {MAX_FILE_SIZE_MB} MB"
                    )

                    return False


            heruntergeladene_bytes = 0

            with temporaerer_pfad.open("wb") as output_file:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if not chunk:
                        continue

                    output_file.write(chunk)

                    heruntergeladene_bytes += len(chunk)

                    if (
                        MAX_FILE_SIZE_MB is not None
                        and heruntergeladene_bytes
                        > MAX_FILE_SIZE_MB
                        * 1024
                        * 1024
                    ):
                        raise ValueError(
                            "Datei überschreitet während "
                            "des Downloads die erlaubte Größe."
                        )


        if heruntergeladene_bytes <= 0:
            raise ValueError(
                "Die heruntergeladene Datei ist leer."
            )


        temporaerer_pfad.replace(zielpfad)

        queue_df.at[
            index,
            "download_status",
        ] = "downloaded"

        queue_df.at[
            index,
            "downloaded_at",
        ] = str(
            pd.Timestamp.now(tz="UTC").isoformat()
        )

        queue_df.at[
            index,
            "local_file_path",
        ] = str(zielpfad)

        queue_df.at[
            index,
            "actual_file_size",
        ] = str(heruntergeladene_bytes)

        queue_df.at[
            index,
            "download_error",
        ] = ""

        return True


    except Exception as error:

        if temporaerer_pfad.exists():
            try:
                temporaerer_pfad.unlink()
            except OSError:
                pass

        queue_df.at[
            index,
            "download_status",
        ] = "failed"

        queue_df.at[
            index,
            "download_error",
        ] = (
            f"{type(error).__name__}: {error}"
        )

        return False


# ============================================================
# 10. OFFENE DOKUMENTE ERMITTELN
# ============================================================

bereits_geladen = int(
    queue_df["download_status"]
    .eq("downloaded")
    .sum()
)

offene_maske = ~queue_df[
    "download_status"
].isin(
    [
        "downloaded",
        "skipped_too_large",
    ]
)

offene_indices = queue_df[
    offene_maske
].index.tolist()


print("\n" + "=" * 78)
print("STATUS VOR DEM START")
print("=" * 78)

print("\nBereits heruntergeladen:")
print(f"{bereits_geladen:,}")

print("\nNoch offen oder fehlgeschlagen:")
print(f"{len(offene_indices):,}")


# ============================================================
# 11. DOWNLOAD-GRENZE DIESES LAUFS
# ============================================================

if MAX_DOWNLOADS_PRO_LAUF is None:
    indices_dieser_lauf = offene_indices
else:
    indices_dieser_lauf = offene_indices[
        :MAX_DOWNLOADS_PRO_LAUF
    ]


print("\nDokumente in diesem Lauf:")
print(f"{len(indices_dieser_lauf):,}")


# ============================================================
# 12. DOWNLOAD STARTEN
# ============================================================

erfolgreich_dieser_lauf = 0
fehlgeschlagen_dieser_lauf = 0

try:

    for position, index in enumerate(
        indices_dieser_lauf,
        start=1,
    ):

        row = queue_df.loc[index]

        dokument_id = int(row["id"])
        titel = str(row["title"])[:70]

        print(
            f"[{position}/{len(indices_dieser_lauf)}] "
            f"ID {dokument_id}: {titel}",
            end=" ... ",
            flush=True,
        )

        erfolgreich = dokument_herunterladen(
            index=index,
            row=row,
        )

        if erfolgreich:
            erfolgreich_dieser_lauf += 1
            print("OK")
        else:
            fehlgeschlagen_dieser_lauf += 1

            status = str(
                queue_df.at[
                    index,
                    "download_status",
                ]
            )

            print(status.upper())


        if position % SAVE_EVERY == 0:

            status_speichern()
            fehler_speichern()

            print(
                f"    Zwischenstand nach "
                f"{position} Dokumenten gespeichert."
            )


        time.sleep(PAUSE_SECONDS)


except KeyboardInterrupt:

    print()
    print("Download wurde manuell unterbrochen.")


finally:

    status_speichern()
    fehler_speichern()
    session.close()


# ============================================================
# 13. ABSCHLUSSSTATISTIK
# ============================================================

gesamt_erfolgreich = int(
    queue_df["download_status"]
    .eq("downloaded")
    .sum()
)

gesamt_fehlgeschlagen = int(
    queue_df["download_status"]
    .eq("failed")
    .sum()
)

gesamt_uebersprungen = int(
    queue_df["download_status"]
    .eq("skipped_too_large")
    .sum()
)

gesamt_offen = int(
    queue_df["download_status"]
    .eq("pending")
    .sum()
)


print("\n" + "=" * 78)
print("DOWNLOAD-LAUF ABGESCHLOSSEN")
print("=" * 78)

print("\nIn diesem Lauf erfolgreich:")
print(f"{erfolgreich_dieser_lauf:,}")

print("\nIn diesem Lauf fehlgeschlagen:")
print(f"{fehlgeschlagen_dieser_lauf:,}")

print("\nInsgesamt erfolgreich:")
print(f"{gesamt_erfolgreich:,}")

print("\nInsgesamt fehlgeschlagen:")
print(f"{gesamt_fehlgeschlagen:,}")

print("\nWegen Größe übersprungen:")
print(f"{gesamt_uebersprungen:,}")

print("\nNoch pending:")
print(f"{gesamt_offen:,}")

print("\nStatusdatei:")
print(STATUS_CSV)

print("\nFehlerdatei:")
print(FEHLER_CSV)

print("\nDokumentordner:")
print(DOKUMENTORDNER)


# ============================================================
# 14. STATUSÜBERSICHT ANZEIGEN
# ============================================================

status_uebersicht = (
    queue_df["download_status"]
    .value_counts(dropna=False)
    .rename_axis("download_status")
    .reset_index(name="anzahl")
)

print("\n" + "=" * 78)
print("STATUSÜBERSICHT")
print("=" * 78)

display(status_uebersicht)

FRAGDENSTAAT-DOKUMENT-DOWNLOADER

Arbeitsordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Datenordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Dokumentordner:
C:\Users\Admin\Desktop\OpenLens\Dokumente

Vorhandene Statusdatei wird geladen ...

Geladene Datei:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_download_status.csv

Datensätze:
258,116

Datentypen der Statusspalten wurden korrigiert.

STATUS VOR DEM START

Bereits heruntergeladen:
200

Noch offen oder fehlgeschlagen:
257,916

Dokumente in diesem Lauf:
100
[1/100] ID 163389: Abschiebung von Kosovo - Albanern ... OK
[2/100] ID 164903: Analphabetismus ... OK
[3/100] ID 166796: Unterstützung für die Bereitschaftspolizei ... OK
[4/100] ID 170241: "Ich bin stolz, Deutscher zu sein" ... OK
[5/100] ID 172592: Folgen für die brandenburgische Wirtschaft aufgrund der Insolvenz der  ... OK
[6/100] ID 171076: Minderjährige unbegleitete Flüchtlinge ... OK
[7/100] ID 167688: Eingriff in das föderale System der Landeszentralbanke

,download_status,anzahl
0,pending,257816
1,downloaded,300
